Import libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

: 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

: 

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully!")

In [ ]:
appointments = pd.read_csv(
    "../data/processed/appointments_cleaned.csv"
)

clinics = pd.read_csv(
    "../data/processed/clinics_cleaned.csv"
)

print("Appointments shape:", appointments.shape)
print("Clinics shape:", clinics.shape)

display(appointments.head())
display(clinics.head())

In [ ]:
appointments = pd.read_csv(
    "../data/processed/appointments_cleaned.csv"
)

clinics = pd.read_csv(
    "../data/processed/clinics_cleaned.csv"
)

print("Appointments shape:", appointments.shape)
print("Clinics shape:", clinics.shape)

display(appointments.head())
display(clinics.head())

In [ ]:
appointments["booked_on"] = pd.to_datetime(
    appointments["booked_on"],
    errors="coerce"
)

appointments["appt_date"] = pd.to_datetime(
    appointments["appt_date"],
    errors="coerce"
)

appointments["no_show"] = pd.to_numeric(
    appointments["no_show"],
    errors="coerce"
)

appointments["lead_days"] = (
    appointments["appt_date"] -
    appointments["booked_on"]
).dt.days

# Merge clinic information
df = appointments.merge(
    clinics,
    on="clinic_id",
    how="left"
)

print("Final EDA dataset:", df.shape)
display(df.head())

In [ ]:
print("Missing values:")
display(df.isnull().sum())

print("\nDuplicate appointment IDs:")
print(df["appointment_id"].duplicated().sum())

print("\nNo-show distribution:")
print(df["no_show"].value_counts(dropna=False))

In [ ]:
total_appointments = len(df)

total_no_shows = df["no_show"].sum()

total_attended = total_appointments - total_no_shows

no_show_rate = (
    total_no_shows / total_appointments
) * 100

attendance_rate = (
    total_attended / total_appointments
) * 100

print("Total Appointments:", total_appointments)
print("Total No-Shows:", total_no_shows)
print("Total Attended:", total_attended)

print("No-Show Rate:", round(no_show_rate, 2), "%")
print("Attendance Rate:", round(attendance_rate, 2), "%")

In [ ]:
total_appointments = len(df)

total_no_shows = df["no_show"].sum()

total_attended = total_appointments - total_no_shows

no_show_rate = (
    total_no_shows / total_appointments
) * 100

attendance_rate = (
    total_attended / total_appointments
) * 100

print("Total Appointments:", total_appointments)
print("Total No-Shows:", total_no_shows)
print("Total Attended:", total_attended)

print("No-Show Rate:", round(no_show_rate, 2), "%")
print("Attendance Rate:", round(attendance_rate, 2), "%")

In [ ]:
status_counts = df["status"].value_counts()

plt.figure(figsize=(8, 5))

plt.bar(
    status_counts.index,
    status_counts.values
)

plt.title("Appointment Status Distribution")
plt.xlabel("Appointment Status")
plt.ylabel("Number of Appointments")

plt.show()

In [ ]:
clinic_analysis = (
    df.groupby(["clinic_name", "city"])
    .agg(
        total_appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

clinic_analysis["no_show_rate"] = (
    clinic_analysis["no_shows"] /
    clinic_analysis["total_appointments"] * 100
).round(2)

clinic_analysis = clinic_analysis.sort_values(
    "no_show_rate",
    ascending=False
)

display(clinic_analysis)

In [ ]:
plt.figure(figsize=(12, 6))

plt.barh(
    clinic_analysis["clinic_name"],
    clinic_analysis["no_show_rate"]
)

plt.xlabel("No-Show Rate (%)")
plt.ylabel("Clinic")
plt.title("Clinic-wise No-Show Rate")

plt.gca().invert_yaxis()

plt.show()

In [ ]:
df["lead_band"] = pd.cut(
    df["lead_days"],
    bins=[0, 7, 14, 30, float("inf")],
    labels=["1-7", "8-14", "15-30", "31+"],
    right=True
)

lead_analysis = (
    df.groupby("lead_band", observed=False)
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

lead_analysis["no_show_rate"] = (
    lead_analysis["no_shows"] /
    lead_analysis["appointments"] * 100
).round(2)

display(lead_analysis)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    lead_analysis["lead_band"].astype(str),
    lead_analysis["no_show_rate"]
)

plt.xlabel("Booking Lead Time (Days)")
plt.ylabel("No-Show Rate (%)")
plt.title("Impact of Booking Lead Time on No-Shows")

plt.show()

In [ ]:
df["month"] = df["appt_date"].dt.to_period("M").astype(str)

monthly_analysis = (
    df.groupby("month")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

monthly_analysis["no_show_rate"] = (
    monthly_analysis["no_shows"] /
    monthly_analysis["appointments"] * 100
).round(2)

display(monthly_analysis)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    monthly_analysis["month"],
    monthly_analysis["no_show_rate"],
    marker="o"
)

plt.title("Monthly No-Show Rate Trend")
plt.xlabel("Month")
plt.ylabel("No-Show Rate (%)")

plt.xticks(rotation=45)
plt.grid(True)

plt.show()

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 18, 30, 45, 60, 100],
    labels=[
        "0-18",
        "19-30",
        "31-45",
        "46-60",
        "60+"
    ],
    include_lowest=True
)

age_analysis = (
    df.groupby("age_group", observed=False)
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

age_analysis["no_show_rate"] = (
    age_analysis["no_shows"] /
    age_analysis["appointments"] * 100
).round(2)

display(age_analysis)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    age_analysis["age_group"].astype(str),
    age_analysis["no_show_rate"]
)

plt.title("No-Show Rate by Patient Age Group")
plt.xlabel("Age Group")
plt.ylabel("No-Show Rate (%)")

plt.show()

In [ ]:
gender_analysis = (
    df.groupby("gender", dropna=False)
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

gender_analysis["no_show_rate"] = (
    gender_analysis["no_shows"] /
    gender_analysis["appointments"] * 100
).round(2)

display(gender_analysis)

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    gender_analysis["gender"].fillna("Missing").astype(str),
    gender_analysis["no_show_rate"]
)

plt.title("No-Show Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("No-Show Rate (%)")

plt.show()

In [ ]:
reminder_analysis = (
    df.groupby("reminder_sent", dropna=False)
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

reminder_analysis["no_show_rate"] = (
    reminder_analysis["no_shows"] /
    reminder_analysis["appointments"] * 100
).round(2)

display(reminder_analysis)

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    reminder_analysis["reminder_sent"].fillna("Missing").astype(str),
    reminder_analysis["no_show_rate"]
)

plt.title("No-Show Rate by Reminder Status")
plt.xlabel("Reminder Sent")
plt.ylabel("No-Show Rate (%)")

plt.show()

In [ ]:
prior_analysis = (
    df.groupby("prior_no_shows", dropna=False)
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

prior_analysis["no_show_rate"] = (
    prior_analysis["no_shows"] /
    prior_analysis["appointments"] * 100
).round(2)

display(prior_analysis)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    prior_analysis["prior_no_shows"].astype(str),
    prior_analysis["no_show_rate"]
)

plt.title("No-Show Rate by Previous No-Show History")
plt.xlabel("Previous No-Shows")
plt.ylabel("Current No-Show Rate (%)")

plt.show()

In [ ]:
df["distance_band"] = pd.cut(
    df["distance_km"],
    bins=[-0.001, 5, 10, 20, float("inf")],
    labels=[
        "0-5 km",
        "5-10 km",
        "10-20 km",
        "20+ km"
    ]
)

distance_analysis = (
    df.groupby("distance_band", observed=False)
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

distance_analysis["no_show_rate"] = (
    distance_analysis["no_shows"] /
    distance_analysis["appointments"] * 100
).round(2)

display(distance_analysis)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    distance_analysis["distance_band"].astype(str),
    distance_analysis["no_show_rate"]
)

plt.title("No-Show Rate by Patient Distance")
plt.xlabel("Distance from Clinic")
plt.ylabel("No-Show Rate (%)")

plt.show()

In [ ]:
channel_analysis = (
    df.groupby("booking_channel", dropna=False)
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

channel_analysis["no_show_rate"] = (
    channel_analysis["no_shows"] /
    channel_analysis["appointments"] * 100
).round(2)

display(channel_analysis)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    channel_analysis["booking_channel"].fillna("Missing").astype(str),
    channel_analysis["no_show_rate"]
)

plt.title("No-Show Rate by Booking Channel")
plt.xlabel("Booking Channel")
plt.ylabel("No-Show Rate (%)")

plt.xticks(rotation=30)

plt.show()

In [ ]:
visit_analysis = (
    df.groupby("visit_type", dropna=False)
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

visit_analysis["no_show_rate"] = (
    visit_analysis["no_shows"] /
    visit_analysis["appointments"] * 100
).round(2)

display(visit_analysis)

In [ ]:
fee_analysis = (
    df.groupby("fee_inr")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

fee_analysis["no_show_rate"] = (
    fee_analysis["no_shows"] /
    fee_analysis["appointments"] * 100
).round(2)

display(fee_analysis)

In [ ]:
eda_summary = {
    "Total Appointments": len(df),

    "Total No-Shows": int(df["no_show"].sum()),

    "Overall No-Show Rate": round(
        df["no_show"].mean() * 100, 2
    ),

    "Highest No-Show Clinic": clinic_analysis.iloc[0]["clinic_name"],

    "Highest Clinic No-Show Rate": clinic_analysis.iloc[0]["no_show_rate"],

    "Highest Lead-Time Band": lead_analysis.sort_values(
        "no_show_rate",
        ascending=False
    ).iloc[0]["lead_band"],

    "Highest Monthly No-Show Rate": monthly_analysis[
        "no_show_rate"
    ].max()
}

for metric, value in eda_summary.items():
    print(f"{metric}: {value}")

In [ ]:
import os

os.makedirs("../reports/eda_results", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)

results = {
    "clinic_analysis": clinic_analysis,
    "lead_time_analysis": lead_analysis,
    "monthly_analysis": monthly_analysis,
    "age_analysis": age_analysis,
    "gender_analysis": gender_analysis,
    "reminder_analysis": reminder_analysis,
    "prior_no_show_analysis": prior_analysis,
    "distance_analysis": distance_analysis,
    "channel_analysis": channel_analysis
}

for name, result in results.items():
    result.to_csv(
        f"../reports/eda_results/{name}.csv",
        index=False
    )

pd.DataFrame(
    list(eda_summary.items()),
    columns=["Metric", "Value"]
).to_csv(
    "../reports/eda_results/eda_summary.csv",
    index=False
)

print("EDA results exported successfully!")

In [ ]:
plt.savefig(
    "../reports/figures/clinic_no_show_analysis.png",
    dpi=300,
    bbox_inches="tight"
)